Intial Claude script to investigate how to handle the data

"""
Step 1: Inspect the structure of the NOAA Global Drifter Program (GDP) hourly
dataset via the clouddrift package.
 
This does NOT download the full 16GB dataset. clouddrift.datasets.gdp1h()
opens the data lazily from a cloud-optimized Zarr store on AWS -- actual
array values are only pulled down when you explicitly compute/index them.
This script just prints metadata and a tiny preview, so it should run
quickly even on a laptop with a normal internet connection.
 
Run this, then paste the full printed output back to me so I can write
Step 2 (the actual per-region EDA) using the confirmed variable names.
"""

In [5]:
import xarray as xr
from clouddrift.datasets import gdp1h
 
print("Opening GDP hourly dataset (lazy load, should be quick)...")
ds = gdp1h()
 
print("\n" + "=" * 80)
print("FULL DATASET REPR")
print("=" * 80)
print(ds)
 
print("\n" + "=" * 80)
print("DIMENSIONS")
print("=" * 80)
print(dict(ds.sizes))
 
print("\n" + "=" * 80)
print("COORDINATES")
print("=" * 80)
for name in ds.coords:
    print(f"  {name}: dims={ds.coords[name].dims}, dtype={ds.coords[name].dtype}")
 
print("\n" + "=" * 80)
print("DATA VARIABLES (name, dims, dtype)")
print("=" * 80)
for name in ds.data_vars:
    v = ds.data_vars[name]
    print(f"  {name}: dims={v.dims}, dtype={v.dtype}")
 
# Try to identify the likely obs-level lat/lon/time/drifter-id variables
print("\n" + "=" * 80)
print("LIKELY KEY VARIABLES (best-guess by name)")
print("=" * 80)
candidates = {
    "latitude": [n for n in list(ds.coords) + list(ds.data_vars) if "lat" in n.lower()],
    "longitude": [n for n in list(ds.coords) + list(ds.data_vars) if "lon" in n.lower()],
    "time": [n for n in list(ds.coords) + list(ds.data_vars) if "time" in n.lower()],
    "drifter/traj id": [n for n in list(ds.coords) + list(ds.data_vars) if "id" in n.lower()],
    "rowsize": [n for n in list(ds.coords) + list(ds.data_vars) if "rowsize" in n.lower()],
}
for label, names in candidates.items():
    print(f"  {label}: {names}")
 
# Small preview of the first ~5 observations for whichever obs-dim lat/lon we find
print("\n" + "=" * 80)
print("SMALL PREVIEW (first 5 values of key obs-level variables, if found)")
print("=" * 80)
obs_dim_vars = [n for n in list(ds.coords) + list(ds.data_vars) if "obs" in ds[n].dims]
for n in obs_dim_vars:
    if any(k in n.lower() for k in ["lat", "lon", "time", "id"]):
        try:
            print(f"  {n}[:5] = {ds[n].isel(obs=slice(0, 5)).values}")
        except Exception as e:
            print(f"  {n}: could not preview ({e})")
 
print("\nDone. Please paste the full output back to me.")

Opening GDP hourly dataset (lazy load, should be quick)...

FULL DATASET REPR
<xarray.Dataset> Size: 16GB
Dimensions:                (traj: 19396, obs: 197214787)
Coordinates:
    id                     (traj) int64 155kB ...
    time                   (obs) datetime64[ns] 2GB ...
Dimensions without coordinates: traj, obs
Data variables: (12/58)
    BuoyTypeManufacturer   (traj) |S20 388kB ...
    BuoyTypeSensorArray    (traj) |S20 388kB ...
    CurrentProgram         (traj) float64 155kB ...
    DeployingCountry       (traj) |S20 388kB ...
    DeployingShip          (traj) |S20 388kB ...
    DeploymentComments     (traj) int16 39kB ...
    ...                     ...
    start_lat              (traj) float32 78kB ...
    start_lon              (traj) float32 78kB ...
    typebuoy               (traj) |S10 194kB ...
    typedeath              (traj) int8 19kB ...
    ve                     (obs) float32 789MB ...
    vn                     (obs) float32 789MB ...
Attributes: (12/18)
  

Step 2: Per-region EDA on the NOAA Global Drifter Program (GDP) hourly dataset.
 
Goal: for a fixed set of lat/lon grid boxes representing major maritime
economic zones, compute summary statistics describing how much drifter
data coverage each region has -- so you can pick the "best" (highest
coverage) region.
 
Approach (memory-conscious, since the full dataset is 197M observations):
  - The dataset is a "ragged array": traj-level `id`/`rowsize` tell us which
    contiguous block of the `obs` dimension belongs to which drifter.
  - We loop over drifters in BATCHES (not one at a time, not all at once),
    pulling only lat/lon/time/ve/vn for that batch, classifying each
    observation into a region (or "Unclassified"), and accumulating running
    per-region statistics. Peak memory stays at "one batch" size, not
    "whole dataset" size.
  - Expect this to take a while (tens of minutes) since it streams ~7GB
    total from the cloud in ~40 chunks. Progress is printed as it goes.
 
Output: a CSV + printed summary table ranking regions by observation count
(= data coverage/density), plus a simple bar chart.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from clouddrift.datasets import gdp1h
 
# ---------------------------------------------------------------------------
# 1. Define the regions (fixed lat/lon grid boxes = major economic zones).
#    Edit this dict freely -- add/remove/resize boxes as needed.
# ---------------------------------------------------------------------------
REGIONS = {
    "Kuroshio Stream"
    "Gulf of Mexico":     {"lat_min": 18, "lat_max": 31, "lon_min": -98, "lon_max": -80},
    "Caribbean Sea":      {"lat_min": 9,  "lat_max": 22, "lon_min": -88, "lon_max": -60},
    "North Sea":          {"lat_min": 51, "lat_max": 61, "lon_min": -4,  "lon_max": 9},
    "Mediterranean Sea":  {"lat_min": 30, "lat_max": 46, "lon_min": -6,  "lon_max": 36},
    "Persian Gulf":       {"lat_min": 24, "lat_max": 30, "lon_min": 48,  "lon_max": 56},
    "Strait of Malacca":  {"lat_min": 1,  "lat_max": 6,  "lon_min": 95,  "lon_max": 104},
    "South China Sea":    {"lat_min": 0,  "lat_max": 23, "lon_min": 99,  "lon_max": 121},
    "East China Sea":     {"lat_min": 24, "lat_max": 33, "lon_min": 120, "lon_max": 130},
    "Bay of Bengal":      {"lat_min": 5,  "lat_max": 22, "lon_min": 80,  "lon_max": 95},
    "Gulf of Guinea":     {"lat_min": -5, "lat_max": 8,  "lon_min": -10, "lon_max": 10},
    "Gulf Stream":        {"lat_min": 25,  "lat_max": 45, "lon_min": -80, "lon_max": -50},
    "Kuroshio Current":   {"lat_min": 24,  "lat_max": 40, "lon_min": 122, "lon_max": 155},
    "Agulhas Current":    {"lat_min": -45, "lat_max": -25, "lon_min": 15, "lon_max": 40},
}
 
BATCH_SIZE = 500  # number of drifters (trajectories) pulled per network request
 
# ---------------------------------------------------------------------------
# 2. Open the dataset (lazy) and get traj-level id/rowsize (cheap, small).
# ---------------------------------------------------------------------------
print("Opening GDP hourly dataset...")
ds = gdp1h()
 
ids = ds["id"].values          # (n_traj,) drifter identifiers
rowsize = ds["rowsize"].values.astype(np.int64)  # (n_traj,) obs count per drifter
n_traj = len(ids)
 
obs_end = np.cumsum(rowsize)
obs_start = obs_end - rowsize  # start offset (inclusive) of each drifter's obs block
 
print(f"Total drifters: {n_traj:,}   Total observations: {obs_end[-1]:,}")
 
# ---------------------------------------------------------------------------
# 3. Accumulators for running per-region statistics.
# ---------------------------------------------------------------------------
acc = {
    name: {
        "n_obs": 0,
        "drifters": set(),
        "sum_speed": 0.0,
        "sumsq_speed": 0.0,
        "n_speed": 0,
        "min_time": None,
        "max_time": None,
    }
    for name in list(REGIONS.keys()) + ["Unclassified"]
}
 
# ---------------------------------------------------------------------------
# 4. Stream through drifters in batches, classify each obs into a region,
#    and update the running accumulators.
# ---------------------------------------------------------------------------
n_batches = int(np.ceil(n_traj / BATCH_SIZE))
print(f"Processing {n_traj:,} drifters in {n_batches} batches of {BATCH_SIZE}...")
 
for b in range(n_batches):
    i0 = b * BATCH_SIZE
    i1 = min(i0 + BATCH_SIZE, n_traj)
 
    obs_slice = slice(int(obs_start[i0]), int(obs_end[i1 - 1]))
    batch = ds.isel(obs=obs_slice)[["lat", "lon", "time", "ve", "vn"]].load()
 
    lat = batch["lat"].values
    lon = batch["lon"].values
    time = batch["time"].values
    ve = batch["ve"].values
    vn = batch["vn"].values
 
    # Build a per-observation drifter index (local to this batch) so we can
    # record which drifters touched which region.
    local_rowsize = rowsize[i0:i1]
    local_ids = ids[i0:i1]
    drifter_idx = np.repeat(np.arange(len(local_ids)), local_rowsize)
 
    speed = np.sqrt(ve.astype(np.float64) ** 2 + vn.astype(np.float64) ** 2)
 
    assigned = np.zeros(len(lat), dtype=bool)
    for name, box in REGIONS.items():
        mask = (
            (lat >= box["lat_min"]) & (lat <= box["lat_max"]) &
            (lon >= box["lon_min"]) & (lon <= box["lon_max"])
        )
        if not mask.any():
            continue
        assigned |= mask
 
        a = acc[name]
        a["n_obs"] += int(mask.sum())
        a["drifters"].update(local_ids[np.unique(drifter_idx[mask])])
 
        sp = speed[mask]
        sp = sp[~np.isnan(sp)]
        a["sum_speed"] += float(sp.sum())
        a["sumsq_speed"] += float((sp ** 2).sum())
        a["n_speed"] += len(sp)
 
        tmin, tmax = time[mask].min(), time[mask].max()
        a["min_time"] = tmin if a["min_time"] is None else min(a["min_time"], tmin)
        a["max_time"] = tmax if a["max_time"] is None else max(a["max_time"], tmax)
 
    # Anything not in any box
    unmask = ~assigned
    if unmask.any():
        a = acc["Unclassified"]
        a["n_obs"] += int(unmask.sum())
 
    if (b + 1) % 5 == 0 or (b + 1) == n_batches:
        print(f"  batch {b + 1}/{n_batches} done "
              f"(drifters {i0}-{i1 - 1}, obs so far processed: {int(obs_end[i1 - 1]):,})")
 
# ---------------------------------------------------------------------------
# 5. Assemble the summary table.
# ---------------------------------------------------------------------------
rows = []
total_obs = int(obs_end[-1])
for name, a in acc.items():
    mean_speed = a["sum_speed"] / a["n_speed"] if a["n_speed"] > 0 else np.nan
    var_speed = (a["sumsq_speed"] / a["n_speed"] - mean_speed ** 2) if a["n_speed"] > 0 else np.nan
    std_speed = np.sqrt(var_speed) if var_speed and var_speed > 0 else np.nan
    span_days = (
        (pd.Timestamp(a["max_time"]) - pd.Timestamp(a["min_time"])).days
        if a["min_time"] is not None else np.nan
    )
    rows.append({
        "region": name,
        "n_obs": a["n_obs"],
        "pct_of_all_obs": 100 * a["n_obs"] / total_obs,
        "n_unique_drifters": len(a["drifters"]) if name != "Unclassified" else np.nan,
        "mean_speed_m_s": mean_speed,
        "std_speed_m_s": std_speed,
        "first_obs": a["min_time"],
        "last_obs": a["max_time"],
        "coverage_span_days": span_days,
    })
 
summary = pd.DataFrame(rows)
 
# Rank regions (excluding "Unclassified") by observation count = data coverage/density
ranked = (
    summary[summary["region"] != "Unclassified"]
    .sort_values("n_obs", ascending=False)
    .reset_index(drop=True)
)
 
print("\n" + "=" * 100)
print("PER-REGION SUMMARY (ranked by observation count = data coverage/density)")
print("=" * 100)
print(ranked.to_string(index=False))
 
unclassified_pct = summary.loc[summary["region"] == "Unclassified", "pct_of_all_obs"].iloc[0]
print(f"\n{unclassified_pct:.1f}% of all observations fell outside every defined region box.")
 
if len(ranked):
    best = ranked.iloc[0]
    print(f"\nHighest-coverage region: {best['region']} "
          f"({best['n_obs']:,} observations, {best['n_unique_drifters']:.0f} unique drifters, "
          f"{best['pct_of_all_obs']:.2f}% of all data)")
 
# ---------------------------------------------------------------------------
# 6. Save outputs.
# ---------------------------------------------------------------------------
summary.to_csv("gdp_region_summary.csv", index=False)
print("\nSaved full summary to gdp_region_summary.csv")
 
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(ranked["region"], ranked["n_obs"])
ax.set_xlabel("Number of observations")
ax.set_title("GDP drifter observation coverage by economic-zone region")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("gdp_region_coverage.png", dpi=150)
print("Saved bar chart to gdp_region_coverage.png")

Opening GDP hourly dataset...
Total drifters: 19,396   Total observations: 197,214,787
Processing 19,396 drifters in 39 batches of 500...
  batch 5/39 done (drifters 2000-2499, obs so far processed: 31,032,121)
  batch 10/39 done (drifters 4500-4999, obs so far processed: 56,381,817)


In [2]:
ds.info()

xarray.Dataset {
dimensions:
	traj = 19396 ;
	obs = 197214787 ;

variables:
	|S20 BuoyTypeManufacturer(traj) ;
		BuoyTypeManufacturer:long_name = Buoy type manufacturer ;
		BuoyTypeManufacturer:units = - ;
	|S20 BuoyTypeSensorArray(traj) ;
		BuoyTypeSensorArray:long_name = Buoy type sensor array ;
		BuoyTypeSensorArray:units = - ;
	float64 CurrentProgram(traj) ;
		CurrentProgram:long_name = Current Program ;
		CurrentProgram:units = - ;
	|S20 DeployingCountry(traj) ;
		DeployingCountry:long_name = Deploying country ;
		DeployingCountry:units = - ;
	|S20 DeployingShip(traj) ;
		DeployingShip:long_name = Name of deployment ship ;
		DeployingShip:units = - ;
	int16 DeploymentComments(traj) ;
		DeploymentComments:long_name = Deployment comments ;
		DeploymentComments:units = - ;
	|S20 DeploymentStatus(traj) ;
		DeploymentStatus:long_name = Deployment status ;
		DeploymentStatus:units = - ;
	float32 DragAreaAboveDrogue(traj) ;
		DragAreaAboveDrogue:long_name = Drag area above drogue. ;
		Dr